# Submission v2-A: Soft voting, v2 features (199)

Self-contained notebook: it starts from the raw competition files (`train.jsonl`, `test.jsonl`,
`type_chart_gen2.csv`, `sample_submission.csv`) and rebuilds everything (preprocessing, move-damage table,
species table, features, model) before writing the submission. It does not read any intermediate file
(`data/processed/`, saved models or `best_params.json`): the tuned parameters are written in the notebook.

The project code is imported from the team repository ([mogliaaaa/FDS-Pokemon-Battles-Prediction-2026-Team-Plasma-Version](https://github.com/mogliaaaa/FDS-Pokemon-Battles-Prediction-2026-Team-Plasma-Version)),
which is cloned when the notebook is not run from inside the repo (e.g. on Kaggle, internet must be on).

Steps: 1. setup, 2. data, 3. preprocessing, 4. features, 5. model, 6. validation (train only),
7. fit on the full train and write the submission.

## 1. Setup

In [ ]:
import subprocess
import sys
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

REPO_URL = "https://github.com/mogliaaaa/FDS-Pokemon-Battles-Prediction-2026-Team-Plasma-Version.git"
REPO_REF = "main"   # pin a commit hash or tag for the final, exactly reproducible run

def find_repo_root():
    # Local run: the notebook is in the repo (root or notebooks/)
    for p in (Path.cwd(), Path.cwd().parent):
        if (p / "src" / "features.py").exists():
            return p.resolve()
    # Kaggle (or anywhere else): clone the repository next to the notebook
    target = Path("PokePredict")
    if not (target / "src" / "features.py").exists():
        subprocess.run(["git", "clone", REPO_URL, str(target)], check=True)
        subprocess.run(["git", "-C", str(target), "checkout", REPO_REF], check=True)
    return target.resolve()

ROOT = find_repo_root()
sys.path.insert(0, str(ROOT))
commit = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print(f"repo: {ROOT} | commit: {commit or 'unknown'}")

from sklearn.ensemble import VotingClassifier

from src import preprocessing as pp
from src.data import find_data_dir, load_jsonl, load_type_chart
from src.features import build_features, robust_columns
from src.models import get_model_specs, params_to_text, with_params
from src.move_damage import fit_move_damage
from src.validation import SEED, cv_scores

RUN_CV = True   # 5-fold CV on train before the final fit (set to False to only write the submission)

## 2. Data

On Kaggle the files are in `/kaggle/input/...`, locally in `data/raw/` (`find_data_dir` tries both).

In [ ]:
DATA_DIR = find_data_dir()
train = load_jsonl(DATA_DIR / "train.jsonl")
test = load_jsonl(DATA_DIR / "test.jsonl")
type_chart = load_type_chart(DATA_DIR / "type_chart_gen2.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
print(f"data dir: {DATA_DIR} | train: {len(train):,} | test: {len(test):,} | sample submission: {len(sample):,}")

## 3. Preprocessing

Same steps as `notebooks/02_preproc.ipynb` (code in `src/preprocessing.py` and `src/move_damage.py`):
- `team_frame`: P1 team and P2 lead, with normalised types and an under-level flag;
- `fit_move_damage`: expected damage of each move estimated from the observed HP drops, so that
  variable-power moves (base power 0 in the data) get a realistic value. **Fitted on train only**;
- `timeline_frame`: one row per battle, side and turn, with the action labels and the expected damage;
- `pokemon_states`: last known state of every Pokémon seen in the 30 turns (KO, HP, vanished);
- `build_species_table`: base stats and types per species, **from train only** (unknown species in test
  get the mean as a fallback).

In [ ]:
t0 = time.time()
damage_table = fit_move_damage(train)
species = pp.build_species_table(train)

def preprocess(battles):
    team = pp.team_frame(battles)
    timeline = pp.timeline_frame(battles, damage_table)
    states = pp.pokemon_states(timeline)
    return team, timeline, states

tables_train = preprocess(train)
tables_test = preprocess(test)
print(f"preprocessing done in {time.time() - t0:.0f}s")

## 4. Features (v2)

The 155 numeric features of `notebooks/03_features.ipynb` (7 groups: static, state, trend, action,
status_effect, boost, matchup) **plus the 44 `team_matchup` features of experiment 06**
(`notebooks/09_features_v2.ipynb`): matchup between the Pokémon still in play on each side after 30 turns
(type coverage, damage index, speed, HP by status, healers, priority / Counter users, late switches).
199 features in total. The species name columns are left out (experiment 03).

In [ ]:
X_train, groups = build_features(*tables_train, species, type_chart, team_matchup=True)
X_test, _ = build_features(*tables_test, species, type_chart, team_matchup=True)
y = X_train.pop("player_won")

CAT = groups["species_cat"]
NUM = [c for c in X_train.columns if c not in CAT]
ROBUST = robust_columns(NUM)

FEATURES = NUM

X_train, X_test = X_train[FEATURES], X_test[FEATURES]
assert list(X_train.columns) == list(X_test.columns)
print(f"train {X_train.shape} | test {X_test.shape} | features used: {len(FEATURES)}")
if len(FEATURES) < len(NUM):
    print("removed (fragile, based on P2's recorded KOs):", sorted(set(NUM) - set(FEATURES)))

## 5. Model

Tuned parameters from experiment 04; why this model was chosen is explained before the submission (section 7).

In [ ]:
# Tuned in notebooks/04_models.ipynb (experiment 04, RandomizedSearchCV, 5-fold CV, seed 42)
PARAMS = {
    'xgboost': {
        'model__colsample_bytree': 0.6856273981493477,
        'model__gamma': 0.010493314645539311,
        'model__learning_rate': 0.012145928554953287,
        'model__max_depth': 6,
        'model__min_child_weight': 1.3386261584543908,
        'model__n_estimators': 720,
        'model__reg_lambda': 3.5416296712651336,
        'model__subsample': 0.5993578407670862,
    },
    'lightgbm': {
        'model__colsample_bytree': 0.6286197472598067,
        'model__learning_rate': 0.006552992991573526,
        'model__min_child_samples': 106,
        'model__n_estimators': 1424,
        'model__num_leaves': 12,
        'model__reg_alpha': 0.6070874110546393,
        'model__reg_lambda': 2.7129089558470034,
        'model__subsample': 0.6629794526009424,
    },
    'catboost': {
        'model__bagging_temperature': 0.15071754396542947,
        'model__depth': 5,
        'model__iterations': 763,
        'model__l2_leaf_reg': 1.14876138417489,
        'model__learning_rate': 0.03898458749305194,
        'model__random_strength': 1.3551287236845648,
    },
    'gradient_boosting': {
        'model__learning_rate': 0.04760153910846178,
        'model__max_depth': 3,
        'model__max_features': None,
        'model__min_samples_leaf': 91,
        'model__n_estimators': 466,
        'model__subsample': 0.9314950036607718,
    },
}

specs = get_model_specs()
for n in PARAMS:
    print(f"{n:18s} {params_to_text(PARAMS[n])}")

model = VotingClassifier([(n, with_params(specs[n], PARAMS[n])) for n in PARAMS], voting="soft", n_jobs=1)
model

## 6. Validation (train only)

Stratified 5-fold CV with `shuffle=True` and seed 42, as in all experiments; every step that learns from
the data is inside the model, so it is refitted on each training fold. The test set is never used.
The reference values from experiment 04 are in section 7 (repeated CV over 3 seeds,
so a single seed can differ by a few tenths of a point).

In [ ]:
if RUN_CV:
    t0 = time.time()
    scores = cv_scores(model, X_train, y)
    print("accuracy per fold:", scores.round(4))
    print(f"5-fold CV: {scores.mean():.4f} ± {scores.std():.4f}  ({time.time() - t0:.0f}s)")
else:
    print("CV skipped (RUN_CV = False)")

## 7. Final fit and submission

**Approach.** The soft-voting ensemble of submission 1 (XGBoost, LightGBM, CatBoost, Gradient Boosting, same
tuned parameters), trained on the **v2 features: the 155 v1 features plus the 44 `team_matchup` features**
of experiment 06 (199 in total).

Why (experiment 06, train only): with the tuned LightGBM the v2 features gained **+0.22 points** in paired
repeated CV (0.6779 → 0.6801, better in 10 of 15 folds). The gain is within the CV noise, but the new
features carry some information the v1 model misses (`sweep` and `pressure` correlate with its residual,
r ≈ 0.03) and LightGBM gives the group 18% of its total gain. This submission tests them on unseen data.

The tuned parameters are those of experiment 04 (tuned on the v1 features); they were not re-tuned on v2.

The model is trained on the **full train set** (10,000 battles) and predicts the test battles. The file
follows `sample_submission.csv` (`battle_id,player_won`). On Kaggle it is written to
`/kaggle/working/submission.csv`; locally to `submissions/submission_v2_main.csv`.

In [ ]:
t0 = time.time()
model.fit(X_train, y)
pred = pd.Series(model.predict(X_test).astype(int), index=X_test.index)
print(f"fitted and predicted in {time.time() - t0:.0f}s")

submission = sample[["battle_id"]].copy()
submission["player_won"] = submission["battle_id"].map(pred)
assert submission["player_won"].notna().all(), "some test battles have no prediction"
submission["player_won"] = submission["player_won"].astype(int)

kaggle_dir = Path("/kaggle/working")
out_path = kaggle_dir / "submission.csv" if kaggle_dir.exists() else ROOT / "submissions" / "submission_v2_main.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(out_path, index=False)
print(f"saved: {out_path} | rows: {len(submission):,}")
print("predicted P1 win rate:", round(submission["player_won"].mean(), 4))
submission.head()